# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np
import warnings

# 경고 메시지 무시 설정
warnings.filterwarnings('ignore')

# 모델 및 데이터 매개변수 설정
B = 2   # 배치 크기
T = 5   # 시간 단계
D = 1   # 특성 수
U = 3   # LSTM 유닛 수

# 무작위 데이터 생성
X = np.random.randn(B, T, D)
print(X.shape)

(2, 5, 1)


# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [2]:
def lstm(return_sequences=False):
    # 입력 및 LSTM 레이어 정의
    inp = Input(shape=(T, D))
    out = LSTM(U, return_sequences=return_sequences)(inp)

    # 모델 생성 및 예측 수행
    model = Model(inputs=inp, outputs=out)
    return model.predict(X)

# return_sequences=False 일 때, 마지막 시간 단계의 출력만 반환
print("---- return_sequences=False ----> last timestep 의 output 만 반환")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)

# return_sequences=True 일 때, 모든 시간 단계에 대한 출력 반환
print("\n---- return_sequences=True ----> 모든 timestep 별 output 출력")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 output 만 반환
1/1 [==============================] - 1s 512ms/step
(2, 3)
[[ 0.02576907 -0.00606291 -0.01183885]
 [ 0.09833541 -0.07199048  0.12875648]]

---- return_sequences=True ----> 모든 timestep 별 output 출력
1/1 [==============================] - 0s 425ms/step
(2, 5, 3)
[[[-0.07601218  0.06493071 -0.04351426]
  [-0.05093041  0.0207109  -0.03458406]
  [ 0.25388223 -0.14446007  0.25174233]
  [ 0.03458932 -0.09578651  0.02217599]
  [ 0.0126913  -0.03725429  0.01053408]]

 [[-0.08447477  0.07754222 -0.04381963]
  [-0.13661404  0.08989628 -0.08198632]
  [-0.1489046   0.08533458 -0.08960754]
  [-0.01937113 -0.0207329  -0.00160071]
  [-0.09305874  0.05795875 -0.04456405]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [3]:
def lstm(return_state=False):
    # 입력 및 LSTM 레이어 정의
    inp = Input(shape=(T, D))
    out = LSTM(U, return_state=return_state)(inp)

    # 모델 생성
    model = Model(inputs=inp, outputs=out)

    # return_state에 따라 출력값 결정
    if return_state:
        # 출력, hidden state, cell state 반환
        o, h, c = model.predict(X)
        print("o :", o.shape)
        print(o)
        print("h :", h.shape)
        print(h)
        print("c :", c.shape)
        print(c)
    else:
        # 출력만 반환
        o = model.predict(X)
        print("o :", o.shape)
        print(o)

# return_state=False 일 때, 출력만 반환
print("---- return_state=False ----> output only")
lstm(return_state=False)

# return_state=True 일 때, 출력과 숨겨진 상태, 셀 상태 모두 반환
print("\n---- return_state=True ----> output, hidden state, cell state all")
lstm(return_state=True)

---- return_state=False ----> output only
1/1 [==============================] - 0s 420ms/step
o : (2, 3)
[[-0.05969164 -0.0694933   0.12372818]
 [-0.10501127 -0.08847862  0.18474402]]

---- return_state=True ----> output, hidden state, cell state all
1/1 [==============================] - 0s 413ms/step
o : (2, 3)
[[ 0.021256   -0.13314143  0.0176857 ]
 [ 0.1345952  -0.28552806  0.00090308]]
h : (2, 3)
[[ 0.021256   -0.13314143  0.0176857 ]
 [ 0.1345952  -0.28552806  0.00090308]]
c : (2, 3)
[[ 0.03941569 -0.25575426  0.0371496 ]
 [ 0.2150673  -0.44926667  0.00215465]]


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [4]:
# T - 시간 단계 (Time Steps): 순환 신경망이 처리할 입력 시퀀스의 길이
# D - 특성 수 (Features): 입력 데이터의 특성(또는 차원)의 수
# U - LSTM 유닛 수 (LSTM Units): LSTM 레이어 내의 유닛(또는 뉴런)의 수
T, D, U

(5, 1, 3)

In [5]:
def bi_lstm(return_sequences=False, return_state=False):
    # 양방향 LSTM 레이어 정의
    inp = Input(shape=(T, D))
    out = Bidirectional(
            LSTM(U, return_state=return_state, return_sequences=return_sequences))(inp)

    # 모델 생성
    model = Model(inputs=inp, outputs=out)

    # return_state 여부에 따라 출력 결정
    if return_state:
        # 출력, 순방향 및 역방향의 hidden state 및 cell state 반환
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :",o.shape)
        print("h1 :", h1.shape)
        print("c1 :", c1.shape)
        print("h2 :", h2.shape)
        print("c2 :", c2.shape)
    else:
        # 출력만 반환
        o = model.predict(X)
        print("o :", o.shape)

# 양방향 LSTM, return_sequences 및 return_state에 따른 출력 확인
print("*** 순방향, 역방향이 concatenate ***")
print("---- return_sequences=False ----> last timestep 의 output 만 반환")
bi_lstm(return_sequences=False, return_state=False)
print()
print("---- return_sequences=True ----> 모든 timestep 별 output 출력")
bi_lstm(return_sequences=True)
print()
print("---- return_sequences=True, return_state=True")
bi_lstm(return_state=True)

*** 순방향, 역방향이 concatenate ***
---- return_sequences=False ----> last timestep 의 output 만 반환


1/1 [==============================] - 1s 856ms/step
o : (2, 6)

---- return_sequences=True ----> 모든 timestep 별 output 출력


1/1 [==============================] - 1s 1s/step
o : (2, 5, 6)

---- return_sequences=True, return_state=True
1/1 [==============================] - 1s 791ms/step
o : (2, 6)
h1 : (2, 3)
c1 : (2, 3)
h2 : (2, 3)
c2 : (2, 3)


# GRU

- cell state 가 없는 것만 LSTM 과 차이

In [6]:
def gru(return_sequences=False, return_state=False):
    # 입력 및 GRU 레이어 정의
    inp = Input(shape=(T, D))
    out = GRU(U, return_state=return_state, return_sequences=return_sequences)(inp)

    # 모델 생성
    model = Model(inputs=inp, outputs=out)

    # return_state 여부에 따라 출력 결정
    if return_state:
        # 출력과 hidden state 반환
        o, h = model.predict(X)
        print("o :", o.shape)
        print("h :", h.shape)
    else:
        # 출력만 반환
        o = model.predict(X)
        print("o :", o.shape)

# GRU 모델의 다양한 출력 형태 확인
print("---- Many-to-One output ----")
gru(return_sequences=False, return_state=False)
print()
print("---- Many-to-Many output ----")
gru(return_sequences=True)
print()
print("---- Sequence-to-Vector output ----")
gru(return_state=True)

---- Many-to-One output ----
1/1 [==============================] - 0s 399ms/step
o : (2, 3)

---- Many-to-Many output ----
1/1 [==============================] - 0s 398ms/step
o : (2, 5, 3)

---- Sequence-to-Vector output ----
1/1 [==============================] - 0s 440ms/step
o : (2, 3)
h : (2, 3)


# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [7]:
def bi_gru(return_sequences=False, return_state=False):
    # 입력 및 양방향 GRU 레이어 정의
    inp = Input(shape=(T, D))
    out = Bidirectional(
            GRU(U, return_state=return_state, return_sequences=return_sequences))(inp)

    # 모델 생성
    model = Model(inputs=inp, outputs=out)

    # return_state 여부에 따라 출력 결정
    if return_state:
        # 출력 및 양 방향의 hidden state 반환
        o, h1, h2 = model.predict(X)
        print("o :", o.shape)
        print("h1 :", h1.shape)
        print("h2 :", h2.shape)
    else:
        # 출력만 반환
        o = model.predict(X)
        print("o :", o.shape)

# 양방향 GRU 모델의 다양한 출력 형태 확인
print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_gru(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_gru(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 된 sequence-to-vector output")
bi_gru(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
1/1 [==============================] - 1s 1s/step
o : (2, 6)

---- 순방향, 역방향이 concatenate 된 many-to-many output
1/1 [==============================] - 1s 889ms/step
o : (2, 5, 6)

---- 순방향, 역방향이 concatenate 된 sequence-to-vector output
1/1 [==============================] - 1s 751ms/step
o : (2, 6)
h1 : (2, 3)
h2 : (2, 3)
